In [12]:
import requests
import json

BASE_URL = "http://localhost:82"
PREFIX = "/bcap"

# Your active credentials
RAW_COOKIE = 'username-localhost-8888="2|1:0|10:1778013652|23:username-localhost-8888|196:eyJ1c2VybmFtZSI6ICI2YzczYThmNGFjY2E0YjIzOTFkZDg4NmY1Njg2YTRkOCIsICJuYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiZGlzcGxheV9uYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiaW5pdGlhbHMiOiAiQVQiLCAiY29sb3IiOiBudWxsfQ==|18fab2f76018c29028bfeef14fcccb33aa5453d4fe73bf39f039b6497ba0700a"; _xsrf=2|7319bcc9|b85949285c2c715d7addfcdbeb20bbbb|1778013652; csrftoken=8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH; bcap_dev=wpp5ck1j3mj6v2ee0yemu8i35q918lta'
RAW_CSRF_TOKEN = "8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json", 
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": f"{BASE_URL}{PREFIX}/",
    "Cookie": RAW_COOKIE,
    "X-CSRFToken": RAW_CSRF_TOKEN
}

# 2. The exact base URL (No UUID, No trailing slash)
post_url = f"{BASE_URL}{PREFIX}/api/resource/process_requirement?format=json"

print(f"Targeting URL: {post_url}")

# 3. Build the payload (Now with the mandatory Requirement Identification field!)
payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "requirement_identification": {
            "tileid": None,
            "aliased_data": {
                # THIS IS THE MISSING FIELD THE SERVER WAS ASKING FOR
                "requirement_identification": {
                    "node_value": "REQ-001", 
                    "display_value": "",
                    "details": []
                },
                "requirement_name": {
                    "node_value": "Jupyter Raw Cookie Requirement",
                    "display_value": "",
                    "details": []
                }
            }
        },
        "sub_requirement": [
            {
                "tileid": None,
                "aliased_data": {
                    "sub_requirement_sort_order": {
                        "node_value": 1,
                        "display_value": "",
                        "details": []
                    },
                    "sub_requirement_name": {
                        "node_value": "Step 1: Test the clean POST",
                        "display_value": "",
                        "details": []
                    },
                    "sub_requirement_description": {
                        "node_value": "We successfully let the server generate its own UUID!",
                        "display_value": "",
                        "details": []
                    }
                }
            }
        ]
    }
}

# 4. Fire the POST request
print("Submitting new record to database using POST...")
response = requests.post(post_url, headers=headers, data=json.dumps(payload), allow_redirects=False)

print(f"\n--- RESULTS ---")
print(f"Status Code: {response.status_code}")

if response.status_code in [200, 201]:
    try:
        saved_data = response.json()
        print("✅ ULTIMATE SUCCESS! The database accepted the record.")
        
        # Extract the ID the server generated for us to build the link
        new_id = saved_data.get("resourceinstanceid")
        if new_id:
            print(f"The server assigned UUID: {new_id}")
            print(f"Go check your UI at: {BASE_URL}{PREFIX}/report/{new_id}")
        else:
            print("Record saved, but could not parse the new UUID from the response.")
            
    except json.JSONDecodeError:
        print("🚨 The server accepted the POST (200 OK), but returned HTML instead of the expected JSON.")
else:
    print("🚨 Something went wrong on the save.")
    print(response.text[:500])

Targeting URL: http://localhost:82/bcap/api/resource/process_requirement?format=json
Submitting new record to database using POST...

--- RESULTS ---
Status Code: 201
✅ ULTIMATE SUCCESS! The database accepted the record.
The server assigned UUID: ddfeb45b-ab2d-4492-8260-1a8679d4a408
Go check your UI at: http://localhost:82/bcap/report/ddfeb45b-ab2d-4492-8260-1a8679d4a408


In [11]:
import requests
import json

BASE_URL = "http://localhost:82"
PREFIX = "/bcap"

# Your active credentials
RAW_COOKIE = 'username-localhost-8888="2|1:0|10:1778013652|23:username-localhost-8888|196:eyJ1c2VybmFtZSI6ICI2YzczYThmNGFjY2E0YjIzOTFkZDg4NmY1Njg2YTRkOCIsICJuYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiZGlzcGxheV9uYW1lIjogIkFub255bW91cyBUaHlvbmUiLCAiaW5pdGlhbHMiOiAiQVQiLCAiY29sb3IiOiBudWxsfQ==|18fab2f76018c29028bfeef14fcccb33aa5453d4fe73bf39f039b6497ba0700a"; _xsrf=2|7319bcc9|b85949285c2c715d7addfcdbeb20bbbb|1778013652; csrftoken=8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH; bcap_dev=wpp5ck1j3mj6v2ee0yemu8i35q918lta'
RAW_CSRF_TOKEN = "8eTXQ39niTUFgXbIIIQBkTupc8rdxTmH"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json", 
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Referer": f"{BASE_URL}{PREFIX}/",
    "Cookie": RAW_COOKIE,
    "X-CSRFToken": RAW_CSRF_TOKEN
}

# --- STEP 1: CREATE THE PROCESS REQUIREMENT (STANDALONE) ---
req_post_url = f"{BASE_URL}{PREFIX}/api/resource/process_requirement?format=json"

sub_requirements_list = []
steps = [
    ("Step 1: Initial Archaeological Consultation", "Consult with BC Archaeological Branch and local First Nations to identify site potential."),
    ("Step 2: Preliminary Field Survey & Mapping", "Conduct an on-site pedestrian field survey and map known archaeological features."),
    ("Step 3: Archaeological Impact Assessment", "Analyze potential developmental impacts on any identified cultural heritage sites."),
    ("Step 4: Mitigative Design Proposal", "Draft a comprehensive mitigative project design to minimize impact on heritage assets."),
    ("Step 5: Final Inspection & Report Submission", "Perform a post-development inspection and submit final paperwork for regulatory sign-off.")
]

for idx, (name, desc) in enumerate(steps, start=1):
    sub_requirements_list.append({
        "tileid": None,
        "aliased_data": {
            "sub_requirement_sort_order": {
                "node_value": idx,
                "display_value": "",
                "details": []
            },
            "sub_requirement_name": {
                "node_value": name,
                "display_value": "",
                "details": []
            },
            "sub_requirement_description": {
                "node_value": desc,
                "display_value": "",
                "details": []
            }
        }
    })

req_payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "requirement_identification": {
            "tileid": None,
            "aliased_data": {
                "requirement_identification": {
                    "node_value": "REQ-BP-2026", 
                    "display_value": "",
                    "details": []
                },
                "requirement_name": {
                    "node_value": "Archaeological Resource Clearance",
                    "display_value": "",
                    "details": []
                }
            }
        },
        "sub_requirement": sub_requirements_list
    }
}

print(f"1. Submitting Process Requirement to: {req_post_url}...")
req_response = requests.post(req_post_url, headers=headers, data=json.dumps(req_payload), allow_redirects=False)

if req_response.status_code not in [200, 201]:
    print(f"🚨 Failed to create Process Requirement. Status: {req_response.status_code}")
    print(f"👉 Django is redirecting us to: {req_response.headers.get('Location')}")
    print(f"👉 Full Headers: {req_response.headers}")
    raise SystemExit("Stopping execution due to API bounce.")

req_data = req_response.json()
process_requirement_uuid = req_data.get("resourceinstanceid")
print(f"✅ Created Process Requirement successfully! UUID: {process_requirement_uuid}\n")


# --- STEP 2: CREATE THE PERMIT APPLICATION (LINKED IMMEDIATELY) ---
permit_post_url = f"{BASE_URL}{PREFIX}/api/resource/permit_application?format=json"

# Constructing the exact Arches-compliant relationship node structure
permit_payload = {
    "resourceinstanceid": None,
    "aliased_data": {
        "application_identification": {
            "tileid": None,
            "aliased_data": {
                "project_name": {
                    "node_value": "Sample BC Development Project (APP-2026-042)",
                    "display_value": "",
                    "details": []
                },
                "application_id": {
                    "node_value": "APP-2026-042",
                    "display_value": "",
                    "details": []
                }
            }
        },
        "application_admin": {
            "tileid": None,
            "aliased_data": {
                "application_requirement": {
                    # WRAP IT IN A LIST WITH THE RESOURCE_ID KEY!
                    "node_value": [
                        {
                            "resourceId": process_requirement_uuid,
                            "ontologyProperty": "",
                            "inverseOntologyProperty": ""
                        }
                    ],
                    "display_value": "",
                    "details": []
                }
            }
        }
    }
}

print(f"2. Submitting Permit Application linked to Process Requirement '{process_requirement_uuid}'...")
permit_response = requests.post(permit_post_url, headers=headers, data=json.dumps(permit_payload), allow_redirects=False)

print(f"\n--- RESULTS ---")
print(f"Status Code: {permit_response.status_code}")

if permit_response.status_code in [200, 201]:
    permit_data = permit_response.json()
    new_permit_uuid = permit_data.get("resourceinstanceid")
    print("✅ ULTIMATE SUCCESS! Linked records successfully written to the database!")
    print(f"👉 View the Process Requirement: {BASE_URL}{PREFIX}/report/{process_requirement_uuid}")
    print(f"👉 View the linked Permit Application: {BASE_URL}{PREFIX}/report/{new_permit_uuid}")
else:
    print("🚨 Something went wrong on the Permit Application save.")
    print(permit_response.text[:1000])

1. Submitting Process Requirement to: http://localhost:82/bcap/api/resource/process_requirement?format=json...
✅ Created Process Requirement successfully! UUID: bd09427a-7077-4a0b-8588-72cf87740d37

2. Submitting Permit Application linked to Process Requirement 'bd09427a-7077-4a0b-8588-72cf87740d37'...

--- RESULTS ---
Status Code: 500
🚨 Something went wrong on the Permit Application save.
DoesNotExist at /bcap/api/resource/permit_application
ResourceInstance matching query does not exist.

Request Method: POST
Request URL: http://localhost/bcap/api/resource/permit_application?format=json
Django Version: 5.2.13
Python Executable: /usr/bin/python3.11
Python Version: 3.11.0
Python Path: ['/web_root/bcap', '/web_root/bcap', '/tmp/debugpy/_vendored/pydevd', '/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '/usr/local/lib/python3.11/dist-packages', '__editable__.arches_querysets-1.1.1b0.finder.__path_hook__', '/usr/lib/python3/dist-packages', '/web_root/bc